<img src="https://industrial.uniandes.edu.co/sites/default/files/imagenes/uniandeslogo.png" alt="Universidad de los Andes" style="float: right; width: 300px; height: auto;">

# run_experiments.ipynb
**Editor:** Juan Diego Heredia Niño  
**Fecha:** Marzo 2026

Ejecuta todas las especificaciones de modelos (Elastic Net, Random Forest, XGBoost) sobre el conjunto de targets de violencia atípica. Las predicciones exportadas son consumidas directamente por `evaluation_report.ipynb`.

## Importaciones y rutas

In [7]:
import warnings
import sys
import pandas as pd
import numpy as np
import yaml
from pathlib import Path
from sklearn.base import clone

warnings.filterwarnings('ignore')

# importar módulos del pipeline
sys.path.insert(0, str(Path('pipeline').resolve()))
from step01_data_loading      import load_data
from step03_temporal_splits   import create_temporal_splits
from step04_feature_preparation import prepare_features
from step05_imputation        import impute_missing
from step06_scaling           import scale_features
from step07_hyperparameter_tuning import tune_hyperparameters
from step07b_tune_rf          import tune_rf
from step07c_tune_xgb         import tune_xgb
from step08_threshold_optimization import optimize_threshold
from step09_evaluation        import evaluate_model
from step10_interpretability  import get_coefficients
from step10b_feature_importance import get_feature_importance
from step11_export            import export_predictions, export_results

In [8]:
with open('paths.yml', 'r') as f:
    paths = yaml.safe_load(f)

processed  = Path(paths['data']['processed'])
model_dir  = Path(paths['outputs']['model'])
model_dir.mkdir(parents=True, exist_ok=True)

## Análisis de leakage

El target `atypical_violence_{var}[t]` = 1 si `{var}[t] > μ + σ` calculados sobre los cuatro trimestres anteriores (`r1`–`r4`). Predecir el trimestre `t` requiere usar únicamente información disponible hasta `t-1`.

### Variables excluidas por leakage (contemporáneas con t)

| Variable | Razón |
|---|---|
| `iacv`, `iacv2`, `ia`, `igc`, `iif` | Índices del trimestre t — son o correlacionan con el target |
| `t_01`–`t_05`, `t_07`–`t_17` | Tasas de crimen del trimestre t — componentes directos del IACV |
| `iacv_n_mean`, `t_XX_n_mean` (sin rezago) | Medias espaciales del trimestre t |
| `ntl_*_q` (todas) | Luces nocturnas del trimestre t — no disponibles al inicio del período |
| `petroleo_crudo_*`, `cafe_arabica_*`, `oro_*` | Precios promedio del trimestre t — no conocidos hasta cerrar el período |
| `covid`, `covid_d` | Casos y muertes COVID del período t |

### Variables incluidas (sin leakage)

- **Rezagos** `*_r1`–`*_r4`: valores en `t-1` hasta `t-4`, todos disponibles al inicio de `t`.
- **Medias espaciales rezagadas** `*_n_mean_r1`–`*_n_mean_r4`: idem para municipios vecinos.
- **Features derivadas** (`_tendencia`, `_volatilidad`, `_pos_relativa`, `_aceleracion`, `_zscore_4q`, `_fue_shock_r1`): construidas exclusivamente a partir de `r1`–`r4`, sin usar ningún valor en `t`.
- **Variables estáticas**: geografía, distancias, cambios socioeconómicos históricos.
- **Demografía**: proyecciones anuales conocidas antes de iniciar el trimestre.
- **Encoding temporal**: `quarter_sin`, `quarter_cos`, `year_normalized`, `trim_desde_paz`, `trim_desde_paz_sq` — deterministas, conocidos al inicio del período.

## Definición de features

In [9]:

# variables con rezagos r1–r4
VARS_CON_LAGS = [
    'iacv', 'iacv2', 'ia', 'igc', 'iif',
    't_01', 't_02', 't_03', 't_04', 't_05',
    't_07', 't_08', 't_09', 't_10', 't_11',
    't_12', 't_13', 't_14', 't_15', 't_16', 't_17',
]

VARS_CON_N_MEAN = [
    'iacv', 'iacv2',
    't_01', 't_02', 't_03', 't_04', 't_05',
    't_07', 't_08', 't_09', 't_10', 't_11',
    't_12', 't_13', 't_14', 't_15', 't_16', 't_17',
]

VARS_DERIVADAS_FULL = [
    'iacv', 'iacv2', 'ia', 'igc', 'iif',
    't_01', 't_02', 't_03', 't_04', 't_05',
]

VARS_DERIVADAS_BASIC = [
    't_07', 't_08', 't_09', 't_10', 't_11',
    't_12', 't_13', 't_14', 't_15', 't_16', 't_17',
]

NTL_BASE_COLS = [
    'ntl_mean_q', 'ntl_median_q', 'ntl_sum_q', 'ntl_sd_q', 'ntl_cv_q',
    'ntl_trend_q', 'ntl_last_q', 'ntl_meses_validos_q', 'ntl_prop_imputado_q',
    'ntl_log_mean_q', 'ntl_log_sum_q', 'ntl_log_median_q',
    'ntl_delta_q', 'ntl_pct_change_q', 'ntl_zscore_q', 'ntl_ma2q',
]

COMM_BASE_COLS = [
    'petroleo_crudo_mean', 'petroleo_crudo_median',
    'cafe_arabica_mean',   'cafe_arabica_median',
    'oro_mean',            'oro_median',
]

# rezagos crudos de violencia e índices
lags_raw    = [f'{v}_r{i}' for v in VARS_CON_LAGS for i in [1, 2, 3, 4]]

# medias espaciales rezagadas
lags_n_mean = [f'{v}_n_mean_r{i}' for v in VARS_CON_N_MEAN for i in [1, 2, 3, 4]]

# features derivadas — set completo (vars principales)
derivadas_full = [
    f'{v}_{t}'
    for v in VARS_DERIVADAS_FULL
    for t in ['tendencia', 'volatilidad', 'pos_relativa', 'aceleracion', 'zscore_4q', 'fue_shock_r1']
]

# features derivadas — solo tendencia y volatilidad (vars secundarias de violencia)
derivadas_basic = [
    f'{v}_{t}'
    for v in VARS_DERIVADAS_BASIC
    for t in ['tendencia', 'volatilidad']
]

# rezagos de NTL (r1–r4) + tendencia y volatilidad derivadas
ntl_lags      = [f'{v}_r{i}' for v in NTL_BASE_COLS for i in [1, 2, 3, 4]]
ntl_derivadas = [f'{v}_{t}' for v in NTL_BASE_COLS for t in ['tendencia', 'volatilidad']]

# rezagos de COVID (r1, r2)
covid_lags = ['covid_r1', 'covid_r2', 'covid_d_r1', 'covid_d_r2']

# rezago de commodities (r1)
comm_lags = [f'{v}_r1' for v in COMM_BASE_COLS]

# variables estáticas y de contexto
estaticas = [
    'indrural', 'areaoficialkm2', 'altura',
    'discapital', 'dismdo', 'disbogota', 'distancia_mercado',
    'delta_ipm', 'delta_nbi', 'delta_ipm_ledu_p',
    'population', 'women_share',
    'dept_code',
]

# encoding temporal — deterministas, conocidos al inicio del trimestre
temporales = [
    'quarter_sin', 'quarter_cos',
    'year_normalized',
    'trim_desde_paz', 'trim_desde_paz_sq',
]

CANDIDATE_FEATURES = (
    lags_raw + lags_n_mean +
    derivadas_full + derivadas_basic +
    ntl_lags + ntl_derivadas +
    covid_lags + comm_lags +
    estaticas + temporales
)

print(f'candidatas totales definidas: {len(CANDIDATE_FEATURES)}')


candidatas totales definidas: 362


## Especificaciones y configuración

Siete targets × tres modelos = 21 experimentos. El dataset `db_sin_jep.parquet` cubre desde 2012Q3; las variables JEP (`ia`, `igc`, `iif` y sus derivadas) tienen NaN antes de 2017Q1 y se imputan con la mediana de entrenamiento.

El balanceo de clases usa `class_weight='balanced'` en Elastic Net y Random Forest, y `scale_pos_weight` automático (n_neg / n_pos) en XGBoost. Los tres son equivalentes en su efecto sobre el gradiente de pérdida.

In [10]:
# targets a predecir
SPECS = {
#    'iacv':  'atypical_violence_iacv',
    'iacv2': 'atypical_violence_iacv2',
    't_01':  'atypical_violence_t_01',
    't_02':  'atypical_violence_t_02',
    't_03':  'atypical_violence_t_03',
    't_04':  'atypical_violence_t_04',
    't_05':  'atypical_violence_t_05',
}

# modelos a correr
MODELS = ['en', 'rf', 'xgb']

# parámetros fijos
RANDOM_STATE   = 42
CV_SCORING     = 'f1'
N_CV_SPLITS    = 5
TRAIN_PROP     = 0.70
VAL_PROP       = 0.15
TEST_PROP      = 0.15

print(f'total experimentos: {len(SPECS)} specs × {len(MODELS)} modelos = {len(SPECS) * len(MODELS)}')

total experimentos: 6 specs × 3 modelos = 18


## Carga de datos

In [11]:
df = load_data(processed / 'db_sin_jep.parquet')

# dept_code: convertir a category para compatibilidad con XGBoost
df['dept_code'] = df['dept_code'].astype('category')

# filtrar solo features candidatas que existan en el dataset
# (maneja diferencias entre db_sin_jep y db_con_jep automáticamente)
FEATURE_COLS = [c for c in CANDIDATE_FEATURES if c in df.columns]
excluidas    = [c for c in CANDIDATE_FEATURES if c not in df.columns]

print(f'\nfeatures disponibles: {len(FEATURE_COLS)}')
if excluidas:
    print(f'no encontradas en el dataset ({len(excluidas)}): {excluidas[:10]}...')

Cargando datos desde: /Users/juandiegoheredianino/Library/CloudStorage/OneDrive-Universidaddelosandes/early-warning-atypical-violence-forecast/data/processed/db_sin_jep.parquet
✓ Datos cargados: 60,617 filas × 465 columnas
Rango temporal : 2012Q3 → 2025Q4
Periodos únicos: 54 | Municipios: 1123

features disponibles: 350
no encontradas en el dataset (12): ['ia_r1', 'ia_r2', 'ia_r3', 'ia_r4', 'igc_r1', 'igc_r2', 'igc_r3', 'igc_r4', 'iif_r1', 'iif_r2']...


## Loop de experimentos

Para cada combinación (spec, modelo):
1. Preparación de features y splits temporales.
2. Imputación por mediana (ajustada solo en train).
3. Escalado solo para Elastic Net (RF y XGBoost no lo requieren).
4. Búsqueda de hiperparámetros con GridSearchCV + TimeSeriesSplit **sobre train únicamente**.
5. Optimización de threshold sobre validación **(out-of-sample respecto al tuneo)**.
6. Refit del modelo final sobre train+val con los hiperparámetros encontrados.
7. Evaluación final en test con el threshold hallado en val.
8. Exportación de predicciones para `evaluation_report.ipynb`.

**Nota RF:** se usa `class_weight=None` para evitar la compresión de probabilidades que produce `class_weight='balanced'` en Random Forest. El desbalance se maneja mediante optimización del threshold.

In [12]:
resumen = []

for spec_name, target_col in SPECS.items():
    print(f'\n{"="*70}')
    print(f'especificacion: {spec_name}  |  target: {target_col}')
    print(f'{"="*70}')

    splits = create_temporal_splits(
        df, time_col='quarter',
        train_prop=TRAIN_PROP, val_prop=VAL_PROP, test_prop=TEST_PROP,
    )

    X, y, fc = prepare_features(
        df,
        target_col      = target_col,
        municipality_col= 'mun_code',
        time_col        = 'quarter',
        feature_cols    = FEATURE_COLS,
    )

    X_train = X[splits['train_mask']]
    X_val   = X[splits['val_mask']]
    X_test  = X[splits['test_mask']]
    y_train = y[splits['train_mask']]
    y_val   = y[splits['val_mask']]
    y_test  = y[splits['test_mask']]

    mun_test = df.loc[splits['test_mask'], 'mun_code'].reset_index(drop=True)
    qtr_test = df.loc[splits['test_mask'], 'quarter'].astype(str).reset_index(drop=True)

    # imputación (fit solo en train)
    X_train, X_val, X_test, _ = impute_missing(X_train, X_val, X_test)

    # versiones escaladas para Elastic Net
    X_train_sc, X_val_sc, X_test_sc, _ = scale_features(X_train, X_val, X_test, fc)

    # train+val para refit final (después de threshold optimization)
    X_tr_val    = pd.concat([X_train,    X_val   ])
    X_tr_val_sc = pd.concat([X_train_sc, X_val_sc])
    y_tr_val    = pd.concat([y_train,    y_val   ])

    for model_name in MODELS:
        exp_name = f'{model_name}_{spec_name}'
        print(f'\n--- {exp_name} ---')

        if model_name == 'en':
            # 1. tuneo sobre train (CV interno en train)
            model_tuned, _ = tune_hyperparameters(
                X_train_sc, y_train,
                scoring      = CV_SCORING,
                n_cv_splits  = N_CV_SPLITS,
                random_state = RANDOM_STATE,
                class_weight = 'balanced',
            )
            # 2. threshold sobre val (out-of-sample)
            threshold, _ = optimize_threshold(model_tuned, X_val_sc, y_val, plot=False)
            # 3. refit sobre train+val con los mismos hiperparámetros
            final_model = clone(model_tuned)
            final_model.fit(X_tr_val_sc, y_tr_val)
            # 4. evaluar en test
            metrics, y_proba, y_pred = evaluate_model(
                final_model, X_test_sc, y_test, threshold, plot=False
            )
            importancia = get_coefficients(final_model, fc, plot=False)

        elif model_name == 'rf':
            # class_weight=None: evita compresión de probabilidades
            model_tuned, _ = tune_rf(
                X_train, y_train,
                scoring      = CV_SCORING,
                n_cv_splits  = N_CV_SPLITS,
                random_state = RANDOM_STATE,
                class_weight = None,
            )
            threshold, _ = optimize_threshold(model_tuned, X_val, y_val, plot=False)
            final_model = clone(model_tuned)
            final_model.fit(X_tr_val, y_tr_val)
            metrics, y_proba, y_pred = evaluate_model(
                final_model, X_test, y_test, threshold, plot=False
            )
            importancia = get_feature_importance(final_model, fc, plot=False)

        elif model_name == 'xgb':
            model_tuned, _ = tune_xgb(
                X_train, y_train,
                scoring      = CV_SCORING,
                n_cv_splits  = N_CV_SPLITS,
                random_state = RANDOM_STATE,
            )
            threshold, _ = optimize_threshold(model_tuned, X_val, y_val, plot=False)
            final_model = clone(model_tuned)
            final_model.fit(X_tr_val, y_tr_val)
            metrics, y_proba, y_pred = evaluate_model(
                final_model, X_test, y_test, threshold, plot=False
            )
            importancia = get_feature_importance(final_model, fc, plot=False)

        export_predictions(
            results_dir    = model_dir,
            experiment_name= exp_name,
            model_type     = model_name,
            target_col     = target_col,
            mun_codes      = mun_test,
            quarters       = qtr_test,
            y_true         = y_test.reset_index(drop=True),
            y_pred         = y_pred,
            y_proba        = y_proba,
            threshold      = threshold,
        )

        export_results(
            results_dir    = model_dir,
            experiment_name= exp_name,
            coeficientes   = importancia,
        )

        resumen.append({
            'experimento': exp_name,
            'modelo':      model_name,
            'spec':        spec_name,
            'target':      target_col,
            **metrics,
        })

print(f'\n{"="*70}')
print('todos los experimentos completados')
print(f'{"="*70}')


especificacion: iacv2  |  target: atypical_violence_iacv2
✓ Splits por proporciones (70%/15%/15%):
  Train: 37 periodos  (2012Q3 → 2021Q3)  41,526 obs (68.5%)
  Val  :  8 periodos  (2021Q4 → 2023Q3)  8,984 obs (14.8%)
  Test :  9 periodos  (2023Q4 → 2025Q4)  10,107 obs (16.7%)
✓ Sin overlap
FEATURES DEL MODELO
  Features seleccionadas : 350
  Target                 : 'atypical_violence_iacv2'  (prevalencia: 18.94%)

  Variables:
      1. iacv_r1
      2. iacv_r2
      3. iacv_r3
      4. iacv_r4
      5. iacv2_r1
      6. iacv2_r2
      7. iacv2_r3
      8. iacv2_r4
      9. t_01_r1
     10. t_01_r2
     11. t_01_r3
     12. t_01_r4
     13. t_02_r1
     14. t_02_r2
     15. t_02_r3
     16. t_02_r4
     17. t_03_r1
     18. t_03_r2
     19. t_03_r3
     20. t_03_r4
     21. t_04_r1
     22. t_04_r2
     23. t_04_r3
     24. t_04_r4
     25. t_05_r1
     26. t_05_r2
     27. t_05_r3
     28. t_05_r4
     29. t_07_r1
     30. t_07_r2
     31. t_07_r3
     32. t_07_r4
     33. t_08_r1
 

/opt/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max


✓ Mejores hiperparámetros:
  C: 0.001
  l1_ratio: 0.1
  Mejor f1 (CV): 0.3469

Top 5 configuraciones:
 param_C  param_l1_ratio  mean_test_score  std_test_score
   0.001             0.1         0.346949        0.025531
   0.010             0.9         0.346049        0.026913
   0.010             0.7         0.345170        0.025965
   0.010             0.5         0.343317        0.024744
   0.010             0.3         0.340518        0.025908
Threshold óptimo (f1): 0.50
  F1: 0.3518 | Precision: 0.2278 | Recall: 0.7721
EVALUACIÓN FINAL EN TEST SET
  AUPRC                 : 0.2083
  AUROC                 : 0.6321
  Balanced_Accuracy     : 0.5970
  F1_Score              : 0.3021
  Precision             : 0.1914
  Recall                : 0.7164
  Cohen_Kappa           : 0.0909
  Threshold             : 0.5000

Classification Report:
              precision    recall  f1-score   support

     Clase 0     0.9070    0.4775    0.6257      8619
     Clase 1     0.1914    0.7164    0.3021  

/opt/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max


✓ Mejores hiperparámetros:
  C: 0.01
  l1_ratio: 0.9
  Mejor f1 (CV): 0.1083

Top 5 configuraciones:
 param_C  param_l1_ratio  mean_test_score  std_test_score
   0.010             0.9         0.108278        0.016812
   0.010             0.5         0.107137        0.015869
   0.010             0.7         0.106903        0.013017
   0.001             0.1         0.106841        0.012639
   0.010             0.3         0.106528        0.013938
Threshold óptimo (f1): 0.85
  F1: 0.2090 | Precision: 0.1595 | Recall: 0.3029
EVALUACIÓN FINAL EN TEST SET
  AUPRC                 : 0.0922
  AUROC                 : 0.7920
  Balanced_Accuracy     : 0.6087
  F1_Score              : 0.1639
  Precision             : 0.1170
  Recall                : 0.2734
  Cohen_Kappa           : 0.1317
  Threshold             : 0.8500

Classification Report:
              precision    recall  f1-score   support

     Clase 0     0.9795    0.9440    0.9614      9840
     Clase 1     0.1170    0.2734    0.1639   

...


✓ Mejores hiperparámetros:
  C: 0.001
  l1_ratio: 0.1
  Mejor f1 (CV): 0.0365

Top 5 configuraciones:
 param_C  param_l1_ratio  mean_test_score  std_test_score
   0.001             0.1         0.036462        0.034031
   0.001             0.3         0.024135        0.030582
   0.001             0.5         0.022822        0.028133
   0.001             0.9         0.019484        0.027613
   0.001             0.7         0.018941        0.029959
Threshold óptimo (f1): 0.85
  F1: 0.0629 | Precision: 0.0329 | Recall: 0.7015
EVALUACIÓN FINAL EN TEST SET
  AUPRC                 : 0.0466
  AUROC                 : 0.7136
  Balanced_Accuracy     : 0.6084
  F1_Score              : 0.0383
  Precision             : 0.0196
  Recall                : 0.9007
  Cohen_Kappa           : 0.0094
  Threshold             : 0.8500

Classification Report:
              precision    recall  f1-score   support

     Clase 0     0.9953    0.3162    0.4799      9956
     Clase 1     0.0196    0.9007    0.0383  

/opt/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.10/site-packa


✓ Mejores hiperparámetros:
  max_depth: 5
  min_samples_leaf: 1
  min_samples_split: 5
  n_estimators: 100
  Mejor f1 (CV): 0.0000

Top 5 configuraciones:
param_max_depth  param_min_samples_leaf  param_min_samples_split  param_n_estimators  mean_test_score  std_test_score
              5                       1                        5                 100              0.0             0.0
             20                       1                        5                 500              0.0             0.0
             20                       1                       10                 100              0.0             0.0
             20                       1                       10                 300              0.0             0.0
             20                       1                       10                 500              0.0             0.0
Threshold óptimo (f1): 0.10
  F1: 0.1265 | Precision: 0.1345 | Recall: 0.1194
EVALUACIÓN FINAL EN TEST SET
  AUPRC                 : 0.0

/opt/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.10/site-packa


✓ Mejores hiperparámetros:
  colsample_bytree: 1.0
  learning_rate: 0.01
  max_depth: 3
  n_estimators: 300
  subsample: 1.0
  Mejor f1 (CV): 0.0285

Top 5 configuraciones:
 param_colsample_bytree  param_learning_rate  param_max_depth  param_n_estimators  param_subsample  mean_test_score  std_test_score
                    1.0                 0.01                3                 300              1.0         0.028518        0.057036
                    1.0                 0.05                3                 100              0.8         0.028169        0.056338
                    1.0                 0.01                3                 100              1.0         0.027561        0.055121
                    0.8                 0.01                3                 300              1.0         0.027533        0.055067
                    0.8                 0.05                3                 100              1.0         0.026136        0.052273
Threshold óptimo (f1): 0.85
  F1: 

## Resumen de resultados

In [13]:
df_resumen = pd.DataFrame(resumen)
cols_show  = ['experimento', 'AUPRC', 'F1_Score', 'Precision', 'Recall', 'Cohen_Kappa', 'Threshold']
print(df_resumen[cols_show].sort_values('AUPRC', ascending=False).to_string(index=False))

experimento    AUPRC  F1_Score  Precision   Recall  Cohen_Kappa  Threshold
   xgb_t_04 0.294066  0.366926   0.243549 0.743636     0.160431       0.50
  xgb_iacv2 0.291595  0.376313   0.279301 0.576582     0.179122       0.55
   xgb_t_01 0.255580  0.336665   0.237621 0.577285     0.161831       0.55
   rf_iacv2 0.253009  0.348964   0.237353 0.658713     0.117680       0.25
    en_t_04 0.248715  0.340079   0.266920 0.468485     0.166768       0.55
   en_iacv2 0.241536  0.335997   0.234112 0.594895     0.107662       0.55
    rf_t_04 0.227946  0.314685   0.231561 0.490909     0.119295       0.25
    rf_t_01 0.212852  0.308703   0.209802 0.584005     0.117538       0.25
    en_t_01 0.208257  0.302111   0.191417 0.716398     0.090861       0.50
   xgb_t_03 0.177501  0.227993   0.138695 0.640152     0.155468       0.55
    en_t_03 0.141222  0.221951   0.132267 0.689394     0.147191       0.65
    rf_t_03 0.140235  0.231111   0.143646 0.590909     0.160556       0.15
   xgb_t_02 0.116945  0.1